In [37]:
import pandas as pd
import numpy as np

In [38]:
# Demand forecast từ M5

demand_df = pd.read_csv(r"/content/drive/MyDrive/Personal project/Paper/agent_demand_summary.csv")

# Supply Chain Dataset

supply_df = pd.read_csv(r"/content/drive/MyDrive/Personal project/Paper/supply_chain_data.csv")

print(demand_df.shape)
print(supply_df.shape)

display(demand_df.head())
display(supply_df.head())

(3049, 5)
(100, 24)


,id,total_forecast_demand,avg_daily_demand,max_daily_demand,demand_std
0,FOODS_1_001_CA_1_validation,26.496971,0.946320,1.181831,0.150282
1,FOODS_1_002_CA_1_validation,13.636966,0.487035,0.676575,0.078049
2,FOODS_1_003_CA_1_validation,21.046046,0.751645,0.938681,0.101824
3,FOODS_1_004_CA_1_validation,21.444924,0.765890,1.069558,0.268603
4,FOODS_1_005_CA_1_validation,28.450723,1.016097,1.411726,0.186391


,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


In [39]:
print(supply_df.columns.tolist())

['Product type', 'SKU', 'Price', 'Availability', 'Number of products sold', 'Revenue generated', 'Customer demographics', 'Stock levels', 'Lead times', 'Order quantities', 'Shipping times', 'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location', 'Lead time', 'Production volumes', 'Manufacturing lead time', 'Manufacturing costs', 'Inspection results', 'Defect rates', 'Transportation modes', 'Routes', 'Costs']


In [40]:
supply_selected = supply_df[
    [
        "Stock levels",
        "Lead times"
    ]
].copy()

supply_selected.head()

,Stock levels,Lead times
0,58,7
1,53,30
2,1,10
3,23,13
4,5,3


In [41]:
# map data
supply_selected = supply_selected.sample(
    n=len(demand_df),
    replace=True,
    random_state=42
).reset_index(drop=True)

agent_df = pd.concat(
    [
        demand_df.reset_index(drop=True),
        supply_selected
    ],
    axis=1
)

agent_df.head()

,id,total_forecast_demand,avg_daily_demand,max_daily_demand,demand_std,Stock levels,Lead times
0,FOODS_1_001_CA_1_validation,26.496971,0.946320,1.181831,0.150282,100,4
1,FOODS_1_002_CA_1_validation,13.636966,0.487035,0.676575,0.078049,90,25
2,FOODS_1_003_CA_1_validation,21.046046,0.751645,0.938681,0.101824,54,29
3,FOODS_1_004_CA_1_validation,21.444924,0.765890,1.069558,0.268603,76,2
4,FOODS_1_005_CA_1_validation,28.450723,1.016097,1.411726,0.186391,41,27


In [42]:
# doi ten cot
agent_df = agent_df.rename(
    columns={
        "Stock levels": "current_inventory",
        "Lead times": "lead_time"
    }
)

agent_df.head()

,id,total_forecast_demand,avg_daily_demand,max_daily_demand,demand_std,current_inventory,lead_time
0,FOODS_1_001_CA_1_validation,26.496971,0.946320,1.181831,0.150282,100,4
1,FOODS_1_002_CA_1_validation,13.636966,0.487035,0.676575,0.078049,90,25
2,FOODS_1_003_CA_1_validation,21.046046,0.751645,0.938681,0.101824,54,29
3,FOODS_1_004_CA_1_validation,21.444924,0.765890,1.069558,0.268603,76,2
4,FOODS_1_005_CA_1_validation,28.450723,1.016097,1.411726,0.186391,41,27


In [43]:
# safety stock
z = 1.65

agent_df["safety_stock"] = (
    z
    * agent_df["demand_std"]
    * np.sqrt(agent_df["lead_time"]))

agent_df["safety_stock"] = (agent_df["safety_stock"].round())

In [44]:
# lead time demand
agent_df["lead_time_demand"] = (
    agent_df["avg_daily_demand"]
    * agent_df["lead_time"]
)

In [45]:
#reorder point
agent_df["reorder_point"] = (
    agent_df["lead_time_demand"]
    + agent_df["safety_stock"]
)

agent_df["reorder_point"] = (
    agent_df["reorder_point"]
    .round())

In [46]:
# decision
agent_df["decision"] = np.where(
    agent_df["current_inventory"]
    <= agent_df["reorder_point"],
    "ORDER",
    "NO_ORDER"
)

In [47]:
#order quantity
TARGET_DAYS = 30

agent_df["target_stock"] = (
    agent_df["avg_daily_demand"]
    * TARGET_DAYS
)

In [48]:
agent_df["order_quantity"] = np.where(
    agent_df["decision"] == "ORDER",
    (
        agent_df["target_stock"]
        - agent_df["current_inventory"]
    ),
    0
)

agent_df["order_quantity"] = (
    agent_df["order_quantity"]
    .clip(lower=0)
    .round()
    .astype(int)
)

In [49]:
#reorder time
agent_df["reorder_time"] = np.where(
    agent_df["decision"] == "ORDER",
    "Order Today",
    f"Within {agent_df['lead_time'].mean():.0f} days"
)

In [50]:
#final recommendation
recommendation_df = agent_df[
    [
        "id",
        "current_inventory",
        "avg_daily_demand",
        "lead_time",
        "safety_stock",
        "reorder_point",
        "decision",
        "order_quantity",
        "reorder_time"
    ]
]

recommendation_df.head()

,id,current_inventory,avg_daily_demand,lead_time,safety_stock,reorder_point,decision,order_quantity,reorder_time
0,FOODS_1_001_CA_1_validation,100,0.946320,4,0.0,4.0,NO_ORDER,0,Within 16 days
1,FOODS_1_002_CA_1_validation,90,0.487035,25,1.0,13.0,NO_ORDER,0,Within 16 days
2,FOODS_1_003_CA_1_validation,54,0.751645,29,1.0,23.0,NO_ORDER,0,Within 16 days
3,FOODS_1_004_CA_1_validation,76,0.765890,2,1.0,3.0,NO_ORDER,0,Within 16 days
4,FOODS_1_005_CA_1_validation,41,1.016097,27,2.0,29.0,NO_ORDER,0,Within 16 days


In [51]:
#save result
recommendation_df.to_csv(
    "inventory_recommendation.csv",
    index=False
)

print(
    "Saved: inventory_recommendation.csv"
)

Saved: inventory_recommendation.csv


In [52]:
from google.colab import files

files.download('inventory_recommendation.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>